In [1]:
import mlflow
import pandas as pd
from sklearn.model_selection import ParameterGrid, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score,precision_score,recall_score
import numpy as np


In [2]:
print('hello')

hello


In [3]:
models = {
    "logistic_regression":{
        "model": LogisticRegression,
        "params": {
            'C':[0.1,1,0.5],
            'solver': ['lbfgs','saga'],
            'random_state': [42]
        }
    },
    "SVM": {
        "model": SVC,
        "params": {
            'kernel':['linear','rbf'],
            'C': [0.1,0.5,1],
            'random_state': [42]
        }
    },
    "Random_Forest":{
        "model": RandomForestClassifier,
        "params":{
            "n_estimators": [100,200,500],
            'random_state': [42]
        }
    }
}

In [4]:
X_train = pd.read_csv('../data/processed/X_train_processed.csv')
y_train = pd.read_csv('../data/processed/y_train_encoded.csv')
X_test = pd.read_csv('../data/processed/X_test_processed.csv')
y_test = pd.read_csv('../data/processed/y_test_encoded.csv')

raw_data = pd.read_csv('../data/raw/addiction_data.csv')

In [5]:
mlflow.set_experiment("Smartphone Addiction Experiment")


<Experiment: artifact_location='file:///c:/End-to-End-Smartphone-Addiction-Prediction/notebooks/mlruns/1', creation_time=1774255934629, experiment_id='1', last_update_time=1774255934629, lifecycle_stage='active', name='Smartphone Addiction Experiment', tags={}, workspace='default'>

In [6]:
train_dataset = mlflow.data.from_pandas(
    raw_data, name="Smart Phone Addiction", targets = "addiction_level"
)

In [7]:
best_model_info = {
    "model": None,
    "val_accuracy_mean": -1,
    "model_name": None,
    "params": None
}

In [8]:
for model_name , config in models.items():
    with mlflow.start_run(run_name = model_name) as parent_run:
        mlflow.log_param("model_type", model_name)
        mlflow.log_input(train_dataset,context="training")

        for params in ParameterGrid(config["params"]):
            run_name = f"{model_name}__" + "__".join(f"{k}={v}" for k,v in params.items())
            with mlflow.start_run(run_name = run_name, nested = True):
                model = config["model"](**params)
                cv_results = cross_validate(
                    model,
                    X_train,y_train,
                    cv=3,
                    scoring = {
                        'accuracy':  'accuracy',
                        'f1':        'f1_weighted',
                        'precision': 'precision_weighted',
                        'recall':    'recall_weighted'
                    }
                )
                model.fit(X_train,y_train)
                test_predictions = model.predict(X_test)

                val_accuracy_mean = np.mean(cv_results['test_accuracy'])


                mlflow.log_params(params)

                # log validation metrics
                for metric in ['accuracy', 'f1', 'precision', 'recall']:
                    scores = cv_results[f'test_{metric}']
                    mlflow.log_metric(f"val_{metric}_mean", np.mean(scores))


                # log test metrics
                mlflow.log_metric("test_accuracy",  accuracy_score(y_test, test_predictions))
                mlflow.log_metric("test_f1",        f1_score(y_test, test_predictions, average="weighted"))
                mlflow.log_metric("test_precision", precision_score(y_test, test_predictions, average="weighted"))
                mlflow.log_metric("test_recall",    recall_score(y_test, test_predictions, average="weighted"))

                if val_accuracy_mean > best_model_info["val_accuracy_mean"]:
                    best_model_info.update({
                        "model": model,
                        "val_accuracy_mean": val_accuracy_mean,
                        "model_name": model_name,
                        "params": params

                    })


c:\Users\User\miniconda3\envs\phonenv\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\Users\User\miniconda3\envs\phonenv\lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example u

In [9]:
y_train.head()

,0
0,2
1,0
2,0
3,1
4,0


In [10]:
best_model_info

{'model': RandomForestClassifier(n_estimators=500, random_state=42),
 'val_accuracy_mean': np.float64(0.5808359133582167),
 'model_name': 'Random_Forest',
 'params': {'n_estimators': 500, 'random_state': 42}}